# Preprocesado de imagen: deteccion de animales
Con un detector COCO (RT-DETR) sobre la foto 1: cuanto de la foto ocupa el animal, cuantos perros/gatos hay, que tan centrado esta y con que confianza se lo ve.

In [ ]:
# Kaggle monta los inputs en otra carpeta cuando el notebook se crea por API: armar ../input/<nombre>
import os, glob
def _buscar(nombre, archivo):
    for prof in range(0, 5):
        for d in glob.glob("/kaggle/input/" + "*/" * prof):
            d = d.rstrip("/")
            if os.path.basename(d).endswith(nombre) and os.path.exists(os.path.join(d, archivo)):
                return d
    raise FileNotFoundError(nombre)
_inputs = {"petfinder-adoption-prediction": "train/train.csv"}
if not all(os.path.exists(f"../input/{n}/{a}") for n, a in _inputs.items()):
    os.makedirs("/tmp/lab/input", exist_ok=True)
    os.makedirs("/tmp/lab/work", exist_ok=True)
    for n, a in _inputs.items():
        if not os.path.exists(f"/tmp/lab/input/{n}"):
            os.symlink(_buscar(n, a), f"/tmp/lab/input/{n}")
    os.chdir("/tmp/lab/work")
D = "../input/petfinder-adoption-prediction"
OUT = "/kaggle/working"

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
fotos = {}
for parte in ("train", "test"):
    fs = sorted(f for f in os.listdir(f"{D}/{parte}_images") if f.endswith("-1.jpg"))
    fotos[parte] = pd.Series([f"{D}/{parte}_images/{f}" for f in fs], index=[f[:-len("-1.jpg")] for f in fs])
    print(parte, len(fotos[parte]), "fotos")

In [ ]:
import torch
dev = "cuda" if torch.cuda.is_available() else "cpu"
try:
    (torch.ones(4, 4, device=dev) @ torch.ones(4, 4, device=dev)).sum().item()
except Exception as e:  # GPU que la version de torch ya no soporta
    print("cuda no anda, sigo en cpu:", e)
    dev = "cpu"
print("torch", torch.__version__, dev, torch.cuda.get_device_name(0) if dev == "cuda" else "")

In [ ]:
from transformers import AutoImageProcessor, AutoModelForObjectDetection
M = "PekingU/rtdetr_r50vd"  # RT-DETR entrenado en COCO (tiene cat y dog)
proc = AutoImageProcessor.from_pretrained(M)
det = AutoModelForObjectDetection.from_pretrained(M).to(dev).eval()
ANIMAL = {i for i, l in det.config.id2label.items() if l in ("cat", "dog")}
print("clases animal:", {i: det.config.id2label[i] for i in ANIMAL})
UMBRAL = 0.7  # caja que cuenta como animal

def features(res, w, h):
    s, l, b = res["scores"].cpu().numpy(), res["labels"].cpu().numpy(), res["boxes"].cpu().numpy()
    es = np.isin(l, list(ANIMAL))
    s, b = s[es], b[es]
    out = {"img_conf_animal": float(s.max()) if len(s) else 0.0, "img_n_animales": int((s >= UMBRAL).sum())}
    fuertes = b[s >= UMBRAL]
    if len(fuertes):
        areas = (fuertes[:, 2] - fuertes[:, 0]) * (fuertes[:, 3] - fuertes[:, 1])
        x0, y0, x1, y1 = fuertes[areas.argmax()]
        out["img_frac_animal"] = float(areas.max() / (w * h))
        out["img_frac_animales_total"] = float(min(areas.sum() / (w * h), 1.0))
        cx, cy = (x0 + x1) / 2 / w - 0.5, (y0 + y1) / 2 / h - 0.5
        out["img_descentrado"] = float(np.sqrt(cx ** 2 + cy ** 2) / np.sqrt(0.5))
    else:  # no se ve ningun perro/gato con confianza: el animal no ocupa la foto
        out.update({"img_frac_animal": 0.0, "img_frac_animales_total": 0.0, "img_descentrado": np.nan})
    return out

B = 16
for parte in ("train", "test"):
    filas = []
    paths = fotos[parte]
    for k in tqdm(range(0, len(paths), B), mininterval=60):
        ims = []
        for i, p in paths.iloc[k:k + B].items():
            try:
                ims.append((i, Image.open(p).convert("RGB")))
            except Exception:
                pass
        with torch.no_grad():
            enc = proc(images=[im for _, im in ims], return_tensors="pt").to(dev)
            out = det(**enc)
        tam = torch.tensor([[im.height, im.width] for _, im in ims])
        res = proc.post_process_object_detection(out, threshold=0.1, target_sizes=tam)
        filas += [dict(features(r, im.width, im.height), PetID=i) for r, (i, im) in zip(res, ims)]
    d = pd.DataFrame(filas).set_index("PetID")
    d.to_parquet(f"{OUT}/img_det_{parte}.parquet")
    print(parte, d.shape, "sin animal detectado:", (d["img_n_animales"] == 0).mean().round(3))
d.describe().T